In [1]:
import logging
import os
import sys

from dotenv import load_dotenv
from matplotlib.ticker import FuncFormatter
from matplotlib import pyplot as plt
from pathlib import Path
from typing import Tuple

import numpy as np
import polars as pl

# Env
pl.Config.set_tbl_rows(10)
load_dotenv("../.env")
INSTALL_PATH = os.environ["INSTALL_PATH"]
URL_AF = os.environ["API_URL_ALPHAFOLD_STRUCTPRED"]
sys.path.append(str(Path(INSTALL_PATH).resolve()))

# Modules
from modules.pymol_align import get_alphafold_structures, pymol_align_pairs

# Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s:%(name)s:%(message)s"
)
logger = logging.getLogger(__name__)

### Steps

1. Apply preliminary filtering criteria
2. Apply PyMOL alignment
3. Lookup how SMARCA4, KPNA2 fare

<br>

### Questions

1. What is the difference between `evalue` and `evalue_foldseek` cols? (`/hpf/projects/CTrost/ajain/IEI_project`)

In [10]:
# ────────────────────────────────────────────────────────────
#      Args
# ────────────────────────────────────────────────────────────

VAR_NAME_LENGTH_ERROR = "Protein_length_difference"
VAR_NAME_LOG_EVALUE = "neg_log(evalue_foldseek)"
THRESHOLD_LENGTH_ERROR = 0.05
THRESHOLD_OVERLAP_RATIO = 0.0
PYMOL_ALIGN_METHOD = "super"
COL_CLUSTER_ID = "cluster_id"
COL_BACT_PROT_ID = "defense_system_protein_id"
COL_HS_PROT_ID = "protein_id"
COL_EVALUE = "evalue_foldseek"

OUT_DIR = Path(f"{INSTALL_PATH}/results/explore_zorya")
OUT_DIR_AF = Path(f"{INSTALL_PATH}/data_local/alphafold_structures")
OUT_DIR_PM = Path(f"{INSTALL_PATH}/data_local/pymol_alignments/{PYMOL_ALIGN_METHOD}")

data_file = f"{INSTALL_PATH}/data_local/zorya/2025-07-27_zorya_annotated.tsv"

gene_names_to_uniprot = {
    "SMARCA4": "A0A2R8Y866",
    "DDX1": "Q92499",
    "BTAF": "O14981",
}

In [8]:
# ────────────────────────────────────────────────────────────
#      In
# ────────────────────────────────────────────────────────────

OUT_DIR.mkdir(exist_ok=True, parents=True)
OUT_DIR_AF.mkdir(exist_ok=True, parents=True)
OUT_DIR_PM.mkdir(exist_ok=True, parents=True)

df = pl.read_csv(
            data_file,
            has_header=True,
            separator="\t",
            schema_overrides={
                "duplicate_count": pl.Int64,
                "rank": pl.Int64,
                "evalue": pl.Float64,
                "cluFlag": pl.Int64,
                "fident": pl.Float64,
                "alnlen": pl.Int64,
                "mismatch": pl.Int64,
                "gapopen": pl.Float64,
                "qstart": pl.Int64,
                "qend": pl.Int64,
                "tstart": pl.Int64,
                "tend": pl.Int64,
                "evalue_foldseek": pl.Float64,
                "bits": pl.Int64,
                "Query_length": pl.Int64,
                "Human_prot_total_length": pl.Int64,
                "Human_domain_percentage": pl.Float64,
                "Query_percentage": pl.Float64,
                "Query_overlap_w_Human_protein": pl.Float64,
                "Target_length": pl.Int64,
                "Bacterial_prot_total_length": pl.Int64,
                "Bacterial_domain_percentage": pl.Float64,
                "Target_percentage": pl.Float64,
                "Target_overlap_w_Bacterial_protein": pl.Float64,
                "Bacterial_Human_Len_Ratio": pl.Float64,
                "Query_Target_Overlap_Length": pl.Float64,
                "Overlap_ratio": pl.Float64,
            },
            infer_schema_length=10000
        )

df

accession_in_sys,type,subtype,species,sys_id,sys_beg,sys_end,duplicate_count,rank,defense_system_protein_id,defense_system_cluster_id,cluster_id,protein_id,evalue,cluFlag,fident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue_foldseek,bits,Reviewed,Entry Name,Protein names,Gene Names,Organism,Gene Names (ordered locus),Gene Names (ORF),Gene Names (primary),Gene Names (synonym),Proteomes,Gene Ontology (biological process),Gene Ontology (cellular component),Gene Ontology (GO),Gene Ontology (molecular function),Gene Ontology IDs,CDD,FunFam,Gene3D,InterPro,PANTHER,Pfam,PROSITE,SMART,RefSeq,Query_range,Query_length,Human_prot_total_length,Human_prot_domain_match,Human_domain_percentage,Query_percentage,Query_overlap_w_Human_protein,Human_domain_note,Human_domain_evidence,Target_range,Target_length,Bacterial_prot_total_length,Bacterial_prot_domain_match,Bacterial_domain_percentage,Target_percentage,Target_overlap_w_Bacterial_protein,Bacterial_domain_note,Bacterial_domain_evidence,Bacterial_Human_Len_Ratio,Query_Target_Overlap_Length,Overlap_ratio
str,str,str,str,str,str,str,i64,i64,str,str,str,str,f64,i64,f64,i64,i64,f64,i64,i64,i64,i64,f64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,str,f64,f64,f64,str,str,str,i64,i64,str,f64,f64,f64,str,str,f64,f64,f64
"""WP_063192462.1""","""Zorya""","""Zorya_TypeI""","""Geobacillus sp. JS12""","""GCF_001592395_NZ_CP014749_Zory…","""GCF_001592395.1_NZ_CP014749_00…","""GCF_001592395.1_NZ_CP014749_00…",1,202,"""A0A142CZW3""","""A0A355UP19""","""A0A355UP19""","""B4DLD8""",0.0,1,0.21,662,519,0.0,277,934,202,863,3.9580e-33,807,"""unreviewed""","""B4DLD8_HUMAN""","""DNA helicase (EC 3.6.4.12)""",null,"""Homo sapiens (Human)""",null,null,null,null,null,"""chromatin organization [GO:000…","""nucleus [GO:0005634]""","""nucleus [GO:0005634]; ATP bind…","""ATP binding [GO:0005524]; DNA …","""GO:0003677; GO:0004386; GO:000…","""cd18668; CD1_tandem_CHD5-9_lik…","""2.40.50.40:FF:000001; chromodo…","""2.40.50.40; -; 2.;""3.40.50.300…","""IPR051493; CHD.;""IPR016197; Ch…","""PTHR46850; CHROMODOMAIN-HELICA…","""PF00385; Chromo; 2.;""PF00271; …","""PS50013; CHROMO_2; 1.;""PS00690…","""SM00298; CHROMO; 2.;""SM00487; …",null,"""277.0..934.0""",658,1014,"""353..396""",100.0,6.69,64.89,"""['Chromo']""","""['ECO:0000259|PROSITE:PS50013'…","""202.0..863.0""",662,874,"""698..851""",100.0,23.26,75.74,"""['Helicase C-terminal']""","""['ECO:0000259|PROSITE:PS51194'…",0.861933,658.0,0.75286
"""WP_063192462.1""","""Zorya""","""Zorya_TypeI""","""Geobacillus sp. JS12""","""GCF_001592395_NZ_CP014749_Zory…","""GCF_001592395.1_NZ_CP014749_00…","""GCF_001592395.1_NZ_CP014749_00…",1,202,"""A0A142CZW3""","""A0A355UP19""","""A0A355UP19""","""B4DLD8""",0.0,1,0.21,662,519,0.0,277,934,202,863,3.9580e-33,807,"""unreviewed""","""B4DLD8_HUMAN""","""DNA helicase (EC 3.6.4.12)""",null,"""Homo sapiens (Human)""",null,null,null,null,null,"""chromatin organization [GO:000…","""nucleus [GO:0005634]""","""nucleus [GO:0005634]; ATP bind…","""ATP binding [GO:0005524]; DNA …","""GO:0003677; GO:0004386; GO:000…","""cd18668; CD1_tandem_CHD5-9_lik…","""2.40.50.40:FF:000001; chromodo…","""2.40.50.40; -; 2.;""3.40.50.300…","""IPR051493; CHD.;""IPR016197; Ch…","""PTHR46850; CHROMODOMAIN-HELICA…","""PF00385; Chromo; 2.;""PF00271; …","""PS50013; CHROMO_2; 1.;""PS00690…","""SM00298; CHROMO; 2.;""SM00487; …",null,"""277.0..934.0""",658,1014,"""353..396""",100.0,6.69,64.89,"""['Chromo']""","""['ECO:0000259|PROSITE:PS50013'…","""202.0..863.0""",662,874,"""698..851""",100.0,23.26,75.74,"""['Helicase C-terminal']""","""['ECO:0000259|PROSITE:PS51194'…",0.861933,658.0,0.75286
"""WP_063192462.1""","""Zorya""","""Zorya_TypeI""","""Geobacillus sp. JS12""","""GCF_001592395_NZ_CP014749_Zory…","""GCF_001592395.1_NZ_CP014749_00…","""GCF_001592395.1_NZ_CP014749_00…",1,202,"""A0A142CZW3""","""A0A355UP19""","""A0A355UP19""","""B4DLD8""",0.0,1,0.21,662,519,0.0,277,934,202,863,3.9580e-33,807,"""unreviewe

In [18]:
# ────────────────────────────────────────────────────────────
#      Check
# ────────────────────────────────────────────────────────────

df_filt = df.filter(
    pl.col(COL_HS_PROT_ID).is_in(gene_names_to_uniprot.values())
)

df_filt["Entry Name"].unique()

Entry Name
str
"""DDX1_HUMAN"""
